# Урок 6.4. Закрепление: все подходы на одной линейке

Финал модуля 6. Новых концепций не будет — будет то, ради чего всё затевалось:
**один замер, где встречаются все способы получить фирменный стиль**, сводный
квиз и финальная практика.

🎯 **В этом уроке:**

- Финальная таблица: zero-shot, few-shot, системный промпт и LoRA — стиль,
  протечки и цена в токенах на одних и тех же отложенных вопросах.
- Сводный квиз по модулю: 10 сценарных вопросов.
- Практика трёх уровней — на этот раз без решений.

## 1. Карта модуля: что у вас теперь есть

| Урок | Главное | Число, которое стоит помнить |
|---|---|---|
| 6.1 | Знания — в RAG, поведение — в веса; LoRA = поправка (alpha/r)·B·A к замороженной базе | few-shot: 661 токен промпта и стиль 0–2/5; LoRA r=16 — 30 млн параметров (<1%), адаптер 57 МБ; полный FT — 46 ГБ |
| 6.2 | Датасет — диалоги без system, loss только на ответах, шаблон чата — родной | max 285 токенов → max_seq_length 512; 15 шагов обучения на бесплатном T4 |
| 6.3 | Modelfile — Dockerfile модели; TEMPLATE живёт в манифесте, а не в GGUF; приёмка = стиль + общие навыки + зазубривание | прототип на SYSTEM: 25/30 и протечка стиля на вопросе про код |
| 6.4 | Все подходы на одной линейке | — сейчас получим |

И инструменты, которые остаются с вами: `style_score` (эталон дословных якорей),
`data/test_questions.jsonl` (отложенный экзамен), `acceptance()` из 6.3
(приёмка с проверкой протечек), `colab-train-lora.ipynb` (фабрика адаптеров).

## 2. Финальный замер

Методика — всё, что мы собрали за модуль, в одном прогоне:

- **Стиль**: `style_score` на 6 отложенных вопросах → максимум 30.
- **Протечка**: те же якоря на 2 вопросах не про хостинг («Париж», код на Python) —
  в идеале 0.
- **Цена запроса**: `prompt_eval_count` — сколько токенов промпта платим за каждый
  вопрос (few-shot и SYSTEM платят вечно, LoRA — нет).

Участники: zero-shot и few-shot из урока 6.1, `pingvin-prompt` из 6.3 и —
если вы привезли артефакт с Colab — дообученная `pingvin-lora`.

In [ ]:
import json
from pathlib import Path

import httpx

OLLAMA_URL = "http://127.0.0.1:11434"
BASE_MODEL = "qwen2.5:3b"
DATA_DIR = Path("data")

STYLE_ANCHORS = ["Поддержка «Пингвин.Хост» на связи", "\U0001F427",
                 "Что происходит:", "Что делать:", "тикет остаётся открытым"]

style_score = lambda text: sum(anchor in text for anchor in STYLE_ANCHORS)

train = [json.loads(line) for line in
         (DATA_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines()]
test_questions = [json.loads(line)["question"] for line in
                  (DATA_DIR / "test_questions.jsonl").read_text(encoding="utf-8").splitlines()]
GENERAL_QUESTIONS = ["Какая столица Франции? Ответь одним словом.",
                     "Напиши функцию на Python, которая переворачивает строку."]

SYSTEM_SHORT = "Ты — сотрудник поддержки хостинга «Пингвин.Хост». Отвечай на вопросы клиентов."
FEW_SHOT = []
for ex in train[:3]:
    FEW_SHOT.append({"role": "user", "content": ex["instruction"]})
    FEW_SHOT.append({"role": "assistant", "content": ex["output"]})

# подход = (модель, как собрать messages из вопроса)
APPROACHES = {
    "zero-shot":     (BASE_MODEL, lambda q: [{"role": "system", "content": SYSTEM_SHORT},
                                             {"role": "user", "content": q}]),
    "few-shot x3":   (BASE_MODEL, lambda q: [{"role": "system", "content": SYSTEM_SHORT},
                                             *FEW_SHOT,
                                             {"role": "user", "content": q}]),
    "SYSTEM в Modelfile": ("pingvin-prompt", lambda q: [{"role": "user", "content": q}]),
    "LoRA в весах":  ("pingvin-lora", lambda q: [{"role": "user", "content": q}]),
}

def chat(model: str, messages: list[dict]):
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": model, "messages": messages, "stream": False,
        "options": {"temperature": 0, "num_predict": 400},
    }, timeout=300)
    response.raise_for_status()
    data = response.json()
    return data["message"]["content"], data.get("prompt_eval_count", 0)

def measure(model: str, build) -> dict:
    """Стиль на отложенных, протечка на общих, средняя цена промпта."""
    style_total, prompt_tokens = 0, []
    for question in test_questions:
        answer, p_tokens = chat(model, build(question))
        style_total += style_score(answer)
        prompt_tokens.append(p_tokens)
    leak_total = sum(style_score(chat(model, build(q))[0]) for q in GENERAL_QUESTIONS)
    return {"style": style_total, "leak": leak_total,
            "prompt_avg": sum(prompt_tokens) // len(prompt_tokens)}

try:
    available = {m["name"] for m in
                 httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]}
    OLLAMA_UP = True
except httpx.HTTPError as exc:
    OLLAMA_UP = False
    print(f"[Ollama недоступна - {exc.__class__.__name__}; финальный замер пропущен]")

In [ ]:
results = {}
if OLLAMA_UP:
    for name, (model, build) in APPROACHES.items():
        if not any(tag.split(":")[0] == model.split(":")[0] for tag in available):
            results[name] = None                      # модели нет (LoRA ещё не привезли)
            continue
        print(f"меряем: {name} ...")
        results[name] = measure(model, build)

    print()
    print(f"{'подход':<20} {'стиль':>8} {'протечка':>9} {'ток/запрос':>11}  цена входа")
    print("-" * 72)
    ENTRY_COST = {"zero-shot": "0", "few-shot x3": "0",
                  "SYSTEM в Modelfile": "минуты", "LoRA в весах": "часы (Colab)"}
    for name, res in results.items():
        if res is None:
            print(f"{name:<20} {'—':>8} {'—':>9} {'—':>11}  "
                  f"[нет модели: привезите GGUF с Colab, урок 6.3]")
        else:
            print(f"{name:<20} {res['style']:>5}/30 {res['leak']:>9} "
                  f"{res['prompt_avg']:>11}  {ENTRY_COST[name]}")

**Разбор таблицы** (мой прогон; ваши числа сдвинутся — GPU-недетерминизм из
урока 3.3, — но порядок сохранится):

- **zero-shot**: 0/30 при 63 токенах промпта. Дёшево и мимо.
- **few-shot ×3**: **худшая сделка в таблице** — самый дорогой промпт (658 токенов,
  в 2.6 раза дороже SYSTEM-подхода!), а стиля — 10/30, нестабильные крохи:
  на одних вопросах модель копирует пол-шаблона, на других игнорирует его целиком.
- **SYSTEM в Modelfile**: 25/30 — лучший результат без обучения, но не эталон
  (якоря теряются на части вопросов), 3 якоря протечки на вопросах не про хостинг,
  и системный промпт из манифеста стоит 252 токена **на каждом запросе** — плата
  видна прямо в prompt_eval_count, хотя в самом запросе — только вопрос.
- **LoRA в весах**: колонка ждёт вашего артефакта с Colab. Ожидание по опыту
  таких дообучений: стиль 28–30/30 при «голом» промпте в ~30 токенов —
  единственный подход, где качество формы **не оплачивается токенами** на
  каждом запросе.

Главный вывод модуля — не «LoRA лучше всех», а **лестница**: сначала промпт
(бесплатно), потом SYSTEM (минуты), и только когда eval показал, что этого мало, —
дообучение (часы). Мы прошли лестницу честно: у нас есть числа для каждой ступени,
и решение «обучать» опирается на замер, а не на моду.

## 3. Сводный квиз

**1. Коллега предлагает: «Дообучим модель на всей нашей вики, и RAG не понадобится». Что ответить?**

<details><summary>Ответ</summary>

Факты в весах устаревают в момент обновления вики, их нельзя процитировать
с источником, а переобучение на каждый апдейт — дорого. Правило модуля:
знания — в RAG, дообучение — для поведения (стиль, формат, тон). Гибрид —
LoRA-стиль + RAG-факты — законный вариант.
</details>

**2. Loss за 15 шагов не сдвинулся с ~2. Куда смотреть в первую очередь?**

<details><summary>Ответ</summary>

На печать первого обучающего примера: цел ли родной шаблон чата (`<|im_start|>...`)
и работает ли маскирование `train_on_responses_only`. Обучение «не тому» — самая
частая причина мёртвого loss на крошечном датасете; гиперпараметры — потом.
</details>

**3. После обучения style_score на вопросах ИЗ train — 30/30. Можно в прод?**

<details><summary>Ответ</summary>

Нельзя судить по train: после 3 эпох там будет идеально даже при полном
зазубривании. Приёмка — только на отложенных вопросах, плюс проверка общих
навыков (протечки) и дословного цитирования чужих ответов.
</details>

**4. Чем LoRA r=16 отличается от r=128 на нашем датасете из 45 примеров — на практике?**

<details><summary>Ответ</summary>

Ёмкостью поправки: r=128 (сотни млн параметров) на 45 примерах почти гарантированно
зазубрит датасет вместо выучивания правила стиля. Для формы хватает r=8–16;
увеличивать ранг стоит только когда eval показал, что модель «не тянет».
</details>

**5. Почему QLoRA-обучение qwen2.5:3b влезает в ~4 ГБ, а LoRA — в ~7–9?**

<details><summary>Ответ</summary>

Разница — замороженная база: у LoRA она лежит в fp16 (6.2 ГБ), у QLoRA — в 4 битах
NF4 (~1.9 ГБ). Обучаемая часть у обеих одинаково крошечная (адаптер + его
градиенты и Adam), поэтому весь выигрыш — на весах базы.
</details>

**6. Создали модель из своего GGUF; в ответах — куски `<|im_start|>`. Диагноз за 10 секунд?**

<details><summary>Ответ</summary>

Забыт TEMPLATE в Modelfile: шаблон чата хранится в манифесте Ollama, из GGUF он
не приезжает. Модель получает голый текст и выводит разметку наружу. Копируйте
TEMPLATE из `ollama show --modelfile qwen2.5:3b`.
</details>

**7. Через полгода `ollama pull qwen2.5:3b` + старый `ADAPTER` = странные ответы без ошибок в логах. Что случилось?**

<details><summary>Ответ</summary>

Тег мутабелен: под `qwen2.5:3b` теперь другой digest, а адаптер математически
привязан к весам, на которых обучался. Поправка легла на «чужие» веса. Поэтому
digest базы хранится рядом с адаптером, а прод предпочитает merged GGUF.
</details>

**8. Дообученный бот вставляет «Что делать:» в ответ на просьбу написать стихотворение. SYSTEM-прототип из 6.3 делал так же. Причины одинаковые?**

<details><summary>Ответ</summary>

Нет. SYSTEM протекает *по построению*: инструкция велит применять шаблон к каждому
ответу. LoRA протекает *от перекоса данных/обучения*: в датасете не было ни одного
«обычного» ответа, а эпох было много. Лечение разное: промпту нужны новые правила
(и новые качели), модели — контрпримеры в датасет и/или меньше эпох.
</details>

**9. SYSTEM-подход тоже «бесплатный по входу». Когда LoRA всё-таки окупается?**

<details><summary>Ответ</summary>

Когда важна дословная стабильность формы (SYSTEM теряет якоря), когда протечки
недопустимы, и когда объём запросов большой: SYSTEM-промпт (сотни токенов) — это
плата prefill'ом за каждый запрос вечно, LoRA переносит эту плату в разовые часы
обучения. Прикиньте по своим цифрам из финальной таблицы.
</details>

**10. Назовите тройку артефактов, из которой ваша модель пересобирается детерминированно, и роль каждого.**

<details><summary>Ответ</summary>

Digest базовой модели (точные веса-основа), датасет в git (чему учили),
адаптер/конфиг обучения (как учили — r, alpha, seed, эпохи). Из тройки
воспроизводится merged GGUF любой квантизации; потеря любого элемента —
переобучение вслепую.
</details>

## 4. Финальная практика

Это ваш выпускной проект модуля — сначала пробуйте сами. Эталоны заданий 2 и 3 лежат
в проекте (сверяться после попытки): `data/counter_examples.jsonl` + `scripts/build_dataset.py`
+ `scripts/acceptance.py` для задания 2 и папка `capstone/json_bot/` для задания 3.
По возрастанию сложности:

1. **Замкните круг.** Прогоните `colab-train-lora.ipynb` на бесплатном T4,
   положите `pingvin-q4_k_m.gguf` в `artifacts/`, перезапустите урок 6.3
   (создание `pingvin-lora` и приёмка) и этот урок — колонка «LoRA в весах»
   оживёт. Сравните с прототипом: побила ли LoRA 25/30? Остались ли протечки?

2. **Вылечите протечку данными.** Добавьте в `train.jsonl` 8–10 контрпримеров —
   обычные вопросы (код, столицы, стихи) с обычными ответами без якорей, —
   пересоберите датасет уроком 6.2, переобучите и добейтесь на приёмке:
   стиль ≥ 28/30 **и** протечка 0. Это настоящая работа ML-инженера: чинить
   поведение данными, а не костылями в промпте.
   *Эталон:* 10 контрпримеров без единого якоря (код, столица, стихи, перевод,
   SQL, git…), `python scripts/build_dataset.py --with-counter` пересобирает train/val
   и проверяет якоря (5/5 у Пингвина, 0/5 у контрпримеров), `python scripts/acceptance.py
   pingvin-lora` выносит вердикт PASS/FAIL по обоим критериям на вопросах, которых
   в контрпримерах нет.

3. **Капстоун: свой бот от и до.** Выберите задачу из своей практики (стиль
   код-ревью вашей команды, JSON-бот из упражнений 6.1–6.3, тон вашей поддержки).
   Соберите 50+ примеров, определите свой `style_score`, пройдите весь конвейер:
   датасет → Colab → GGUF → Modelfile → приёмка. Финальный аккорд — отдайте
   модель наружу через `ollama-proxy` из урока 5.7: у вас уже есть и модель
   со своим характером, и прод-обвязка для неё. Курс сошёлся.
   *Эталон:* JSON-бот из упражнений 6.1–6.3 в `capstone/json_bot/`: генератор датасета
   (865 примеров: одна confidence на факт, ~30 % отказов, мутации формулировок, экзамен
   без пересечений с обучением), `TASK = "json"` в Colab-ноутбуке, `Modelfile.pingvin-json`,
   `acceptance.py` со своим style_score (JSON парсится, без прозы, схема, типы, смысл по
   виду вопроса) и сравнение с грамматикой `--format-json`.

## 🎓 Итоги модуля 6

- **Матрица выбора**: промпт → few-shot → RAG → fine-tuning; знания — в RAG,
  поведение — в веса. Решение принимает eval (у нас — style_score на отложенных).
- **LoRA**: поправка `(alpha/r)·B·A` к замороженным весам; r=16 на все linear —
  <1% параметров, адаптер 57 МБ. **QLoRA** кладёт базу в 4 бита — обучение 3B
  влезает в бесплатный Colab.
- **Датасет решает**: диалоги с родным шаблоном чата, loss только на ответах,
  train/val + отложенные вопросы; 45 примеров × 3 эпохи = 15 шагов.
- **Возвращение домой**: merge в fp16 → GGUF → квант один раз → Modelfile
  (TEMPLATE — из манифеста базы!) → `ollama create` → digest.
- **Приёмка**: стиль на отложенных, общие навыки без протечек, отсутствие
  зазубривания. Прототип на SYSTEM — обязательный конкурент: 25/30 без единой
  минуты обучения.
- **Эксплуатация**: тройка «digest + датасет + адаптер», candidate-теги, канарейка.

**Дальше — модуль 7 «Мульти-агентные системы»:** одна модель научилась говорить
нашим голосом; теперь научим несколько моделей работать командой — роли,
оркестрация, передача контекста и цена этой архитектуры.

## 📚 Что почитать

- [LoRA](https://arxiv.org/abs/2106.09685) и [QLoRA](https://arxiv.org/abs/2305.14314) — первоисточники
- [Unsloth docs](https://docs.unsloth.ai/) — актуальные рецепты дообучения
- [Ollama: Modelfile](https://docs.ollama.com/modelfile) и [импорт моделей](https://docs.ollama.com/import)
- [The Novice's LLM Training Guide](https://rentry.org/llm-training) — обзор всей кухни дообучения
